In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [7]:
!git clone https://github.com/THU-MIG/RepViT.git
%cd RepViT

Cloning into 'RepViT'...
remote: Enumerating objects: 374, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 374 (delta 81), reused 64 (delta 64), pack-reused 272 (from 1)
Receiving objects: 100% (374/374), 20.74 MiB | 39.33 MiB/s, done.
Resolving deltas: 100% (156/156), done.
/kaggle/working/RepViT


In [8]:
!pip install -r requirements.txt
!pip install timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 22.6 MB/s eta 0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=0c61f12c4117a920d01876ffaea3b0aa194287824c4240e950f14f329eef54ff
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=cb271604117346345edc65b7cdf343bba35c29c784732032906d279364eea223
  Stored in directory: /root/.cache/pip/wheels/7c/96/04/4f5f31ff812f684f69f40cb1634357812220aac58d4698048c
Successfully built fvcore iopath
  Attempting uninstall: timm
    Found existing installation: timm 1.0.25
    Uninstalling timm-1.0.25:
      Succes

In [9]:
!mkdir pretrain
!wget https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m0_9_distill_300e.pth -P pretrain

--2026-04-16 06:33:49--  https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m0_9_distill_300e.pth
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/667632599/0fb45a90-f855-40fe-b501-de1d2a459b0e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-16T07%3A26%3A01Z&rscd=attachment%3B+filename%3Drepvit_m0_9_distill_300e.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-04-16T06%3A25%3A19Z&ske=2026-04-16T07%3A26%3A01Z&sks=b&skv=2018-11-09&sig=XVW99gT9PSdaZe%2FEs30G44LsuuTjeqo6w2dLSW9Vfo4%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NjMyMzAzMCwibmJmIjoxNzc2MzIxMjMwLCJwYXRoIjoicmVsZWFzZWFzc

In [13]:
!ls /kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val | head

ILSVRC2012_val_00000001.JPEG
ILSVRC2012_val_00000002.JPEG
ILSVRC2012_val_00000003.JPEG
ILSVRC2012_val_00000004.JPEG
ILSVRC2012_val_00000005.JPEG
ILSVRC2012_val_00000006.JPEG
ILSVRC2012_val_00000007.JPEG
ILSVRC2012_val_00000008.JPEG
ILSVRC2012_val_00000009.JPEG
ILSVRC2012_val_00000010.JPEG
ls: write error: Broken pipe


In [12]:
!ls /kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train/n01440764 | head

n01440764_10026.JPEG
n01440764_10027.JPEG
n01440764_10029.JPEG
n01440764_10040.JPEG
n01440764_10042.JPEG
n01440764_10043.JPEG
n01440764_10048.JPEG
n01440764_10066.JPEG
n01440764_10074.JPEG
n01440764_10095.JPEG
ls: write error: Broken pipe


In [14]:
import pandas as pd

csv_path = "/kaggle/input/competitions/imagenet-object-localization-challenge/LOC_val_solution.csv"
df = pd.read_csv(csv_path)

df.head()

,ImageId,PredictionString
0,ILSVRC2012_val_00048981,n03995372 85 1 499 272
1,ILSVRC2012_val_00037956,n03481172 131 0 499 254
2,ILSVRC2012_val_00026161,n02108000 38 0 464 280
3,ILSVRC2012_val_00026171,n03109150 0 14 216 299
4,ILSVRC2012_val_00008726,n02119789 255 142 454 329 n02119789 44 21 322 ...


In [15]:
val_img_path = "/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val"
csv_path = "/kaggle/input/competitions/imagenet-object-localization-challenge/LOC_val_solution.csv"

In [17]:
import os
import shutil

val_img_path = "/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val"

output_val = "/kaggle/working/imagenet/val"
os.makedirs(output_val, exist_ok=True)

In [18]:
for _, row in df.iterrows():
    img_name = row["ImageId"] + ".JPEG"
    label = row["PredictionString"].split()[0]  # first label only

    class_dir = os.path.join(output_val, label)
    os.makedirs(class_dir, exist_ok=True)

    src = os.path.join(val_img_path, img_name)
    dst = os.path.join(class_dir, img_name)

    if os.path.exists(src):
        shutil.copy(src, dst)

In [19]:
import os

train_src = "/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train"
train_dst = "/kaggle/working/imagenet/train"

os.makedirs("/kaggle/working/imagenet", exist_ok=True)

# create symbolic link (VERY FAST, saves disk & time)
if not os.path.exists(train_dst):
    os.symlink(train_src, train_dst)

In [22]:
DATA_PATH = "/kaggle/working/imagenet"

In [25]:
!python main.py \
    --eval \
    --model repvit_m0_9 \
    --resume pretrain/repvit_m0_9_distill_300e.pth \
    --data-path /kaggle/working/imagenet

Not using distributed mode
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Creating model: repvit_m0_9
number of params: 5489328
/usr/local/lib/python3.12/dist-packages/timm/utils/cuda.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler()
Creating teacher model: regnety_160
Downloading: "https://dl.fbaipublicfiles.com/deit/regnety_160-a5fe301d.pth" to /root/.cache/torch/hub/checkpoints/regnety_160-a5fe301d.pth
100%|████████████████████████